# FORCE compatible versions (prevents GRPOTrainer crash)

import sys
import subprocess

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

pip_install(["--upgrade", "--force-reinstall",
             "transformers==4.52.0",
             "trl==0.16.1"])

pip_install([
    "accelerate>=0.30.0",
    "datasets",
    "peft",
    "bitsandbytes"
])

print("Installed compatible TRL + Transformers versions")



In [ ]:
import os
os._exit(0)


In [ ]:
# Cell 0.5: Reproducibility + model selection — set SEED and (optionally) MODEL
# To run a different seed, change SEED below before running any other cell.
# Suggested seeds for the reproducibility study: 42 (original), 100, 200, 300.
# Each seed produces an independent training_scores.json + training_log.json.
# Save them as runs/seed_42/, runs/seed_100/, etc., then aggregate via
# scripts/aggregate_seeds.py.
#
# To run a different BASE MODEL for cross-model comparison, set environment
# variables before launching the kernel:
#     LEGALOOM_MODEL_NAME='google/gemma-2-2b-it' LEGALOOM_MODEL_TAG='gemma2b'
# Default is Qwen2.5-3B-Instruct, which produces the existing pinned artifacts.
# See scripts/aggregate_models.py and the Reproducibility section of README.md
# for the full multi-model workflow.

SEED = 42

import os, random

# Model selection — read from environment, default to Qwen2.5-3B
MODEL_NAME = os.environ.get('LEGALOOM_MODEL_NAME', 'Qwen/Qwen2.5-3B-Instruct')
MODEL_TAG  = os.environ.get('LEGALOOM_MODEL_TAG',  'qwen3b')

def seed_all(seed: int) -> None:
    """Best-effort determinism across the ML stack used in this notebook.

    All ML imports are lazy so this cell can safely run BEFORE the install
    cell — numpy/torch/transformers may not be available on first execution.
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass  # numpy not installed yet — Cell 1 will install it
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    try:
        from transformers import set_seed
        set_seed(seed)
    except ImportError:
        pass

seed_all(SEED)
print(f'✓ All available RNGs seeded with SEED={SEED}')
print(f'  MODEL_NAME = {MODEL_NAME}')
print(f'  MODEL_TAG  = {MODEL_TAG}')
print('  (numpy/torch/transformers seeds applied if those libraries are installed;')
print('   re-run this cell after Cell 1 if you want full coverage.)')

In [ ]:
# Cell 1: Detect platform + install
import os

ON_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
print(f'Platform: {"Colab" if ON_COLAB else "HF Space / Local"}')

# Known-good combinations for CUDA 12.4+ / Triton 3.x environments.
# Colab T4: still on CUDA 12.x with older Triton — Unsloth path works.
# HF Space (Nov 2026+): ships with PyTorch 2.5+ / Triton 3.x / CUDA 13 — needs newer bnb.
if ON_COLAB:
    # Colab T4 / A10G: Unsloth path (handles version negotiation)
    !pip install -q unsloth 'transformers>=4.49,<4.55' 'trl>=0.15,<0.17' \
        'peft>=0.13,<0.16' 'datasets>=2.20.0' \
        openenv-core==0.2.3 fastapi uvicorn pydantic httpx openai pyyaml matplotlib
else:
    # HF Space: vanilla HF + PEFT path. Use bitsandbytes>=0.46.1 which has
    # CUDA 13 and Triton 3.x compatibility. The older 0.45.0 pin from
    # earlier runs is broken on the current HF Space image (triton.ops missing,
    # no libbitsandbytes_cuda130.so).
    !pip install -q 'transformers>=4.49,<4.55' 'trl>=0.15,<0.17' \
        'peft>=0.13,<0.16' 'bitsandbytes>=0.46.1' 'accelerate>=1.2.1' \
        'datasets>=2.20.0' \
        openenv-core==0.2.3 fastapi uvicorn pydantic httpx \
        openai pyyaml matplotlib

print('✓ Installed')

# Verify CUDA + bitsandbytes are actually working before continuing
import torch
print(f'  torch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if not ON_COLAB:
    try:
        import bitsandbytes as bnb
        print(f'  bitsandbytes: {bnb.__version__}')
        # Sanity check: trigger a CUDA op to confirm the binary loaded
        if torch.cuda.is_available():
            x = torch.randn(8, 8, device='cuda')
            _ = (x @ x).sum().item()
            print('  ✓ CUDA + bitsandbytes working')
    except Exception as e:
        print(f'  ⚠ bitsandbytes import issue: {e}')
        print('  → Cell 3 will fall back to fp16 if 4-bit quantization fails')

# Re-seed after install
try:
    seed_all(SEED)
    print('✓ Re-seeded after install')
except NameError:
    pass


In [ ]:
# Cell 2: Clone repo into writable location
import sys, os, subprocess

if ON_COLAB:
    WORK_DIR = '/content/legaloom_env'
else:
    WORK_DIR = '/data/legaloom-env' if os.path.exists('/data') and os.access('/data', os.W_OK) else os.path.expanduser('~/legaloom-env')

if not os.path.exists(WORK_DIR):
    subprocess.run(
        ['git', 'clone', 'https://huggingface.co/spaces/aarav0202/legaloom-env', WORK_DIR],
        check=True,
    )

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f'✓ Working dir: {os.getcwd()}')
print(f'✓ Files: {sorted(os.listdir("."))[:8]} ...')

In [ ]:
# Cell 3: Load model — Unsloth on Colab, vanilla HF+PEFT on HF Space
# === PARAMETERIZED VERSION — supports MODEL_NAME / MODEL_TAG from SEED cell ===
# Defaults to Qwen2.5-3B-Instruct. To run a different model, set
# LEGALOOM_MODEL_NAME and LEGALOOM_MODEL_TAG env vars before launching the kernel.
# See Cell 0.5 (SEED + model selection) for details.
print(f'PATCH MARKER MULTIMODEL-2026-04-26 — loading {MODEL_NAME} (tag: {MODEL_TAG})')
import torch, importlib

if ON_COLAB:
    # Unsloth path. Try the requested model first; if Unsloth doesn\'t have an
    # accelerated version, fall back to standard HF + PEFT.
    try:
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_NAME,
            max_seq_length=2048, dtype=None, load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model, r=16,
            target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
            lora_alpha=16, lora_dropout=0, bias='none',
            use_gradient_checkpointing='unsloth', random_state=SEED,
        )
        print(f'  Loaded via Unsloth: {MODEL_NAME}')
    except Exception as e:
        print(f'  ⚠ Unsloth load failed for {MODEL_NAME}: {type(e).__name__}: {str(e)[:120]}')
        print(f'  → Falling back to vanilla HF + PEFT path')
        ON_COLAB = False  # force the fallback branch below

if not ON_COLAB:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # CRITICAL: V3 (working) uses pad_token = eos_token. Other configurations
    # break multi-step rollout — model emits valid step-1 JSON but downstream
    # steps fail, causing all-floor baseline scores. This is the V1/V3 pattern;
    # holds for Qwen, Gemma, and Llama (all three default to pad_token=None).
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    else:
        # Already has a pad token — force it to match eos for multi-step rollout
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # 4-bit with bf16 fallback
    use_4bit = False
    try:
        from transformers import BitsAndBytesConfig
        import bitsandbytes  # noqa: F401
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config,
            device_map='auto', torch_dtype=torch.bfloat16,
        )
        use_4bit = True
        print(f'  Loaded {MODEL_NAME} with 4-bit quantization')
    except Exception as e:
        print(f'  ⚠ 4-bit unavailable ({type(e).__name__}: {str(e)[:80]})')
        print('  → Falling back to bfloat16')
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, device_map='auto', torch_dtype=torch.bfloat16,
        )

    if use_4bit:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable()

    # LoRA target modules differ slightly per architecture, but q/k/v/o + MLP
    # is the universal subset. Qwen2.5, Gemma-2, and Llama-3 all expose
    # gate_proj/up_proj/down_proj for the MLP. PEFT will skip any module not
    # present in the base model rather than erroring.
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_dropout=0, bias='none', task_type='CAUSAL_LM',
    ))

print(f'✓ Model loaded: {MODEL_NAME}')
print(f'  dtype={next(model.parameters()).dtype}')
print(f'  pad_token={tokenizer.pad_token!r} (id={tokenizer.pad_token_id})')
print(f'  eos_token={tokenizer.eos_token!r} (id={tokenizer.eos_token_id})')

assert tokenizer.pad_token_id == tokenizer.eos_token_id, (
    f"BUG: pad_token_id ({tokenizer.pad_token_id}) != eos_token_id "
    f"({tokenizer.eos_token_id}). This will break multi-step rollout."
)
print(f'  ✓ pad_token == eos_token check passed')

# Chat-template sanity check — each model uses a different template, and
# tokenizer.apply_chat_template() picks up the right one automatically when
# you load the tokenizer. This print confirms it.
ct_demo = tokenizer.apply_chat_template(
    [{"role": "user", "content": "test"}],
    tokenize=False, add_generation_prompt=True,
)
print(f'  Chat template demo (first 200 chars):')
print(f'    {ct_demo[:200]!r}')

# Generation sanity check
test_input = tokenizer('Hello, how are', return_tensors='pt').to(model.device)
with torch.no_grad():
    test_out = model.generate(
        **test_input, max_new_tokens=20, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
test_text = tokenizer.decode(test_out[0][test_input.input_ids.shape[1]:], skip_special_tokens=True)
print(f'  Generation sanity check: "Hello, how are" → "{test_text[:60]}"')

# Force-reload train_grpo to ensure we use the patched rollout_episode
import sys
mods_to_reload = [m for m in list(sys.modules.keys()) if m.startswith(("train_grpo", "server."))]
for mod in mods_to_reload:
    del sys.modules[mod]
print(f'  Cleared {len(mods_to_reload)} cached modules — fresh imports next cell')


In [ ]:
# Cell 3.5: VERBOSE DEBUG — call the REAL rollout_episode and inspect the trajectory
# Force-reload modules in case kernel has old versions cached
import sys
for mod in list(sys.modules):
    if mod.startswith('train_grpo') or mod.startswith('server.') or mod.startswith('models'):
        del sys.modules[mod]

import inspect, train_grpo
src_check = inspect.getsource(train_grpo.rollout_episode)
has_invoice_text = 'invoice_text' in src_check
has_invoice_block = 'INVOICE:' in src_check or 'invoice_block' in src_check
fix_present = has_invoice_text and has_invoice_block
print(f'train_grpo.__file__: {train_grpo.__file__}')
print(f'  has invoice_text reference: {has_invoice_text}')
print(f'  has INVOICE block injection: {has_invoice_block}')
print(f'  → invoice_text fix present: {fix_present}')
if not fix_present:
    raise RuntimeError('train_grpo.py is missing the invoice_text injection.')

print('PATCH MARKER C3D4E5 — running real rollout_episode debug')

from server.legaloom_env_environment import LegaloomEnvironment
from train_grpo import rollout_episode

if ON_COLAB:
    from unsloth import FastLanguageModel as FLM
    FLM.for_inference(model)
else:
    model.eval()

env = LegaloomEnvironment()
result = rollout_episode(
    model, tokenizer, env,
    task_id='task_easy',
    seed=42,
    temperature=0.1,
    max_new_tokens=512,
)

print(f'\n=== Rollout finished ===')
print(f'  final_reward: {result["final_reward"]}')
print(f'  steps_used:   {result["steps_used"]}')
print(f'  success:      {result.get("success")}')
print(f'\n=== Trajectory ({len(result["trajectory"])} steps) ===')
for i, step in enumerate(result['trajectory'], 1):
    print(f'\n--- Step {i} ---')
    if step.get('action') is None:
        print(f'  GENERATION FAILED — generated: {step.get("generated", "")[:200]!r}')
        print(f'  error: {step.get("error")}')
    else:
        print(f'  action:        {step["action"]}')
        print(f'  reward:        {step.get("reward")}')
        print(f'  done:          {step.get("done")}')
        if step.get('action_result'):
            print(f'  action_result: {step["action_result"][:200]}')
        if step.get('error'):
            print(f'  ERROR:         {step["error"]}')

# If we got 0.05 final reward, dig into why
if result['final_reward'] <= 0.05:
    print(f'\n⚠ Final reward at floor (0.05). Diagnostic:')
    if not result['trajectory']:
        print('  No trajectory recorded — model never generated any action.')
    else:
        last = result['trajectory'][-1]
        if last.get('action') is None:
            print(f'  Last step: model output unparseable')
        elif last.get('error'):
            print(f'  Last step: env rejected action with error: {last["error"]}')
        elif not last.get('done'):
            print(f'  Last step: episode hit max_steps without submit_answer')
        else:
            print(f'  Last step terminated with reward {last.get("reward")}')


In [ ]:
# Cell 4: Baseline measurement — 30 episodes per task (seeds 42–71)
from train_grpo import rollout_episode
from server.legaloom_env_environment import LegaloomEnvironment

if ON_COLAB:
    from unsloth import FastLanguageModel as FLM
    FLM.for_inference(model)
else:
    model.eval()

print('Measuring baseline (untrained), 30 episodes per task...')
baseline_scores = {}
baseline_per_episode = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])

    mean = sum(scores) / len(scores)
    std = (sum((s - mean) ** 2 for s in scores) / len(scores)) ** 0.5
    baseline_scores[task_id] = mean
    baseline_per_episode[task_id] = scores

    # Early-stop heuristic: if every score is the floor (0.050), the model is
    # not generating valid output. No point burning compute on more tasks.
    n_floor = sum(1 for s in scores if abs(s - 0.05) < 1e-4)
    print(f'  {task_id}: mean={mean:.3f}  std={std:.3f}  ({n_floor}/{len(scores)} at floor)')
    if n_floor == len(scores) and task_id == 'task_easy':
        print('  ⚠ All baseline easy scores are at floor (0.05).')
        print('     This means the model is emitting empty or unparseable output.')
        print('     Common causes: bitsandbytes broken, model failed to load, prompt issue.')
        print('     Stop and re-check Cell 3 model load before continuing.')

print(f'\nBaseline avg: {sum(baseline_scores.values()) / 4:.3f}')


In [ ]:
# Cell 5: Single-phase GRPO on task_hard (40 steps, num_generations=8)
import os
from train_grpo import build_training_dataset, episode_reward_fn
from trl import GRPOConfig, GRPOTrainer

if ON_COLAB:
    from unsloth import FastLanguageModel as FLM, is_bfloat16_supported
    FLM.for_training(model)
    bf16 = is_bfloat16_supported()
    fp16 = not bf16
    gc_kwargs = {}
    batch_size = 1
else:
    model.train()
    model.enable_input_require_grads()
    bf16 = True
    fp16 = False
    gc_kwargs = {'gradient_checkpointing_kwargs': {'use_reentrant': False}}
    batch_size = 8  # HF Space: batch must be >= num_generations

log_history = []
dataset = build_training_dataset(task_ids=['task_hard'], examples_per_task=120, base_seed=SEED)

def _make_reward_fn(tid):
    def fn(prompts, completions, **kwargs):
        clean_kwargs = {k: v for k, v in kwargs.items() if k != 'task_id'}
        return episode_reward_fn(prompts, completions, task_id=tid, **clean_kwargs)
    return fn

cfg_dict = dict(
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=4 if ON_COLAB else 1,
    num_generations=8,
    max_prompt_length=1536,
    max_completion_length=512,
    beta=0.01,
    logging_steps=1,
    max_steps=40,
    output_dir='./legaloom_grpo_output/task_hard',
    report_to='none',
    seed=SEED,
    bf16=bf16,
    fp16=fp16,
    save_strategy='no',
    gradient_checkpointing=True,
)
cfg_dict.update(gc_kwargs)
cfg = GRPOConfig(**cfg_dict)

trainer = GRPOTrainer(
    model=model, args=cfg, train_dataset=dataset,
    reward_funcs=[
    _make_reward_fn('task_easy'),
    _make_reward_fn('task_medium'),
    _make_reward_fn('task_hard'),
    _make_reward_fn('task_expert'),
],
    tokenizer=tokenizer,
)
trainer.train()

for entry in trainer.state.log_history:
    entry['phase'] = 0
    entry['phase_task_id'] = 'task_hard'
log_history.extend(trainer.state.log_history)
print(f'\n✓ Done. {len(log_history)} log entries.')

# === Reward-variance diagnostic ===
# TRL writes frac_reward_zero_std on each train log entry. Steps where this is
# ~1.0 produced zero GRPO advantage (all generations scored identically).
# A high fraction means num_generations is too low for the reward distribution
# OR the reward function is too sparse. Goal: <30% with num_generations=8.
zero_std_steps = [e for e in trainer.state.log_history if e.get('frac_reward_zero_std', 0.0) >= 0.99]
total_steps = [e for e in trainer.state.log_history if 'frac_reward_zero_std' in e]
if total_steps:
    frac = len(zero_std_steps) / len(total_steps)
    print(f'\n=== Reward variance diagnostic ===')
    print(f'Zero-variance training steps: {len(zero_std_steps)}/{len(total_steps)} ({frac:.0%})')
    if frac > 0.5:
        print(f'⚠ More than half of steps wasted. num_generations may need to go higher,')
        print(f'  or the reward distribution is too bimodal for GRPO to grip.')
    elif frac > 0.2:
        print(f'  Some wasted steps but training had useful gradient signal on most steps.')
    else:
        print(f'✓ Strong gradient signal — num_generations=8 mostly produced reward variance.')
else:
    print('(No frac_reward_zero_std field in log_history — TRL version may not emit it.)')

In [ ]:
# Cell 5.5: DIAGNOSTIC — verify training actually updated the model
import torch

print('=== LoRA weight check (B matrices init to zero — nonzero = learned) ===')
for name, param in model.named_parameters():
    if 'lora_B' in name and 'q_proj' in name:
        norm = param.data.norm().item()
        maxv = param.data.abs().max().item()
        status = '✓ LEARNED' if norm > 0.001 else '✗ NO LEARNING (weights still ~zero)'
        print(f'  {name}: norm={norm:.6f}, max={maxv:.6f} → {status}')
        break

print()
print('=== Quick generation test: trained vs base ===')
test_prompt = 'You are a TDS compliance agent. Read the invoice and determine TDS.'
with torch.no_grad():
    inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
    out_trained = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    text_trained = tokenizer.decode(out_trained[0], skip_special_tokens=True)

try:
    with model.disable_adapter():
        out_base = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
        text_base = tokenizer.decode(out_base[0], skip_special_tokens=True)
    same = text_trained == text_base
    print(f'  Outputs identical: {same}')
    if same:
        print('  ⚠ Model may not have learned — outputs identical to base')
    else:
        print('  ✓ Model produces different output — training had effect')
except Exception as e:
    print(f'  Could not compare: {e}')
    print('  Proceeding — LoRA weight norm is the primary check.')

In [ ]:
# Cell 6: Plot reward curves
import matplotlib.pyplot as plt
import json

with open('training_log.json', 'w') as f:
    json.dump(log_history, f, indent=2, default=str)

rewards = [e['reward'] for e in log_history if 'reward' in e]
losses = [e['loss'] for e in log_history if 'loss' in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = list(range(1, len(rewards) + 1))
ax.plot(x, rewards, 'b-', linewidth=1.5, alpha=0.6, label='Episode reward')
if len(rewards) >= 3:
    w = 3
    ma = [sum(rewards[max(0,j-w+1):j+1])/min(j+1,w) for j in range(len(rewards))]
    ax.plot(x, ma, 'r-', linewidth=2.5, label='3-step moving avg')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xlabel('Training Step'); ax.set_ylabel('Episode Reward')
_ml = MODEL_LABELS.get(MODEL_TAG, MODEL_TAG) if 'MODEL_LABELS' in dir() else MODEL_TAG
ax.set_title(f'GRPO — task_hard (40 steps, num_gen=8) — {_ml}')
ax.set_ylim(0, 1); ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
if losses:
    ax2.plot(range(1, len(losses)+1), losses, 'g-', linewidth=1.5, alpha=0.8)
ax2.set_xlabel('Training Step'); ax2.set_ylabel('Loss')
ax2.set_title('GRPO — Loss'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
suffixed_curves = f'reward_curves_{MODEL_TAG}.png'
plt.savefig(suffixed_curves, dpi=150, bbox_inches='tight')
if MODEL_TAG == 'qwen3b':
    plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Save training_log alongside the chart so multi-model runs preserve it
import json as _json
suffixed_log = f'training_log_{MODEL_TAG}.json'
with open(suffixed_log, 'w') as _f:
    _json.dump(log_entries, _f, indent=2)
if MODEL_TAG == 'qwen3b':
    with open('training_log.json', 'w') as _f:
        _json.dump(log_entries, _f, indent=2)

print(f'✓ {suffixed_curves} + {suffixed_log} saved (model={MODEL_TAG})')
if MODEL_TAG == 'qwen3b':
    print(f'  also wrote untagged: reward_curves.png, training_log.json')

In [ ]:
# Cell 7: Trained model evaluation — 30 episodes per task (seeds 42–71)
if ON_COLAB:
    FLM.for_inference(model)
else:
    model.eval()

# Verify adapter is active
if hasattr(model, 'active_adapters'):
    print(f'Active adapters: {model.active_adapters}')

print('Measuring trained model, 30 episodes per task...')
trained_scores = {}
trained_per_episode = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    trained_scores[task_id] = sum(scores) / len(scores)
    trained_per_episode[task_id] = scores
    std = (sum((s - trained_scores[task_id])**2 for s in scores) / len(scores))**0.5
    delta = trained_scores[task_id] - baseline_scores[task_id]
    rel = (delta / baseline_scores[task_id] * 100) if baseline_scores[task_id] > 0 else 0
    print(f'  {task_id}: before={baseline_scores[task_id]:.3f}  after={trained_scores[task_id]:.3f}  std={std:.3f}  Δ={delta:+.3f} ({rel:+.0f}%)')

print(f'\nBaseline avg: {sum(baseline_scores.values()) / 4:.3f}')
print(f'Trained avg:  {sum(trained_scores.values()) / 4:.3f}')
print(f'Lift:         {(sum(trained_scores.values()) - sum(baseline_scores.values())) / 4:+.3f}')

In [ ]:
# Cell 8: Before/After bar chart + save scores (multi-model aware)
import matplotlib.pyplot as plt
import json

# Map MODEL_TAG to a friendly label for chart titles
MODEL_LABELS = {
    'qwen3b': 'Qwen2.5-3B',
    'gemma2b': 'Gemma-2-2B',
    'llama3b': 'Llama-3.2-3B',
}
model_label = MODEL_LABELS.get(MODEL_TAG, MODEL_TAG)

tasks = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
labels = ['Easy', 'Medium', 'Hard', 'Expert']
before = [baseline_scores[t] for t in tasks]
after = [trained_scores[t] for t in tasks]

fig, ax = plt.subplots(figsize=(11, 6))
x = range(len(tasks)); w = 0.36
before_std = [(sum((s - baseline_scores[t])**2 for s in baseline_per_episode[t]) / len(baseline_per_episode[t]))**0.5 for t in tasks]
after_std  = [(sum((s - trained_scores[t])**2  for s in trained_per_episode[t])  / len(trained_per_episode[t]))**0.5  for t in tasks]
ax.bar([i - w/2 for i in x], before, w, yerr=before_std, capsize=4, label='Before GRPO (untrained)', color='#e76f51', error_kw={'alpha': 0.6})
ax.bar([i + w/2 for i in x], after,  w, yerr=after_std,  capsize=4, label='After GRPO (single-phase task_hard)', color='#2a9d8f', error_kw={'alpha': 0.6})
for i in x:
    ax.text(i - w/2, before[i] + 0.015, f'{before[i]:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, after[i]  + 0.015, f'{after[i]:.3f}',  ha='center', fontsize=9)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('Average Score (30 episodes)')
ax.set_title(f'LegaLoom-Env: Before vs After GRPO Training — {model_label}')
ax.set_ylim(0, 1); ax.legend(loc='upper left')
plt.tight_layout()

# Multi-model artifact naming:
# - Always write tag-suffixed copies (training_scores_qwen3b.json, before_after_qwen3b.png)
# - When MODEL_TAG == 'qwen3b' (the default single-model workflow), ALSO write
#   the untagged versions (training_scores.json, before_after.png) so existing
#   downstream scripts (statistical_analysis.py, populate_results.py) keep working.
suffixed_chart = f'before_after_{MODEL_TAG}.png'
suffixed_scores = f'training_scores_{MODEL_TAG}.json'

plt.savefig(suffixed_chart, dpi=150, bbox_inches='tight')
if MODEL_TAG == 'qwen3b':
    plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
plt.show()

scores_payload = {
    'baseline': baseline_scores,
    'trained': trained_scores,
    'baseline_per_episode': baseline_per_episode,
    'trained_per_episode': trained_per_episode,
    'n_episodes_per_task': len(baseline_per_episode['task_easy']),
    'training_config': 'single-phase task_hard, 40 steps, num_gen=8',
    'seed': SEED,
    'model_name': MODEL_NAME,
    'model_tag': MODEL_TAG,
    'model_label': model_label,
}
with open(suffixed_scores, 'w') as f:
    json.dump(scores_payload, f, indent=2)
if MODEL_TAG == 'qwen3b':
    with open('training_scores.json', 'w') as f:
        json.dump(scores_payload, f, indent=2)

print(f'✓ {suffixed_chart} + {suffixed_scores} saved (model={model_label})')
if MODEL_TAG == 'qwen3b':
    print(f'  also wrote untagged: before_after.png, training_scores.json '
          f'(default single-model workflow compatibility)')

In [ ]:
# Cell 9: Print README table + download artifacts
import os

print('=' * 60)
print('PASTE THIS INTO README.md Results section:')
print('=' * 60)
print()
print('| Task | Baseline | After GRPO | Δ |')
print('|------|---------:|-----------:|------:|')
for tid, lbl in [('task_easy','`task_easy`'),('task_medium','`task_medium`'),
                  ('task_hard','`task_hard`'),('task_expert','`task_expert`')]:
    b, a = baseline_scores[tid], trained_scores[tid]
    d = ((a-b)/b*100) if b > 0 else 0
    bold = '**' if a > b else ''
    print(f'| {lbl} | {b:.3f} | {bold}{a:.3f}{bold} | {d:+.0f}% |')
ab = sum(baseline_scores.values())/4
aa = sum(trained_scores.values())/4
ad = ((aa-ab)/ab*100) if ab > 0 else 0
print(f'| **Average** | **{ab:.3f}** | **{aa:.3f}** | **{ad:+.0f}%** |')
print()
print('=' * 60)
print(f'Artifacts in: {os.getcwd()}')
for fname in ['reward_curves.png', 'before_after.png', 'training_scores.json', 'training_log.json']:
    exists = '✓' if os.path.exists(fname) else '✗'
    print(f'  {exists} {fname}')
print()

# Download on Colab, manual on HF Space
if ON_COLAB:
    from google.colab import files
    for fname in ['reward_curves.png', 'before_after.png', 'training_scores.json', 'training_log.json']:
        try:
            files.download(fname)
            print(f'  Downloaded: {fname}')
        except Exception as e:
            print(f'  Download failed: {fname}: {e}')
else:
    print('HF Space: right-click files in JupyterLab browser → Download')
    print('Then: Settings → Hardware → CPU basic to stop GPU billing.')